In [1]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No CUDA')

2.4.1+cu121
12.1
True
Tesla P100-PCIE-16GB


In [2]:
%pip install "transformers>=4.51.0,<5.0.0" sentence-transformers --upgrade --quiet


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip show sentence-transformers transformers | grep -E "Name|Version"

Name: sentence-transformers
Version: 5.5.1
Name: transformers
Version: 4.57.6
Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch, transformers, sentence_transformers
print(torch.__version__)
print(transformers.__version__)      # doit être 4.51.x ou 4.52.x
print(sentence_transformers.__version__)

/home/infres/lelabed-25/projects/ProjectDSAI2026_13/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.4.1+cu121
4.57.6
5.5.1


# Imports

In [5]:
import os, re, sys
sys.path.insert(0, "..")

import torch
import pandas as pd
from pathlib import Path
import glob
from tqdm import tqdm
from dotenv import load_dotenv
from huggingface_hub import login

from candidates.bm25_retriever import BM25Retriever
from candidates.dense_retriever import DenseRetriever
from utils.prompt_builder import build_mcq_prompt 
from utils.response_parser import parse_mcq_response 
from utils.metrics import is_correct, recall_at_k, compute_metrics, print_metrics
from utils.wiki_descriptions import fetch_descriptions_batch
from utils.llm import load_model, generate_batch
from utils.elgold_parser import load_benchmark

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
print(HF_TOKEN is not None)

True


In [7]:
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [8]:
MODE = "zero_shot_bm25"
MODEL_NAME = "Qwen/Qwen3-4B"
K = 100
MAX_EXAMPLES = None      
BATCH_SIZE = 64

MAX_NEW_TOKENS = 2048

TITLES_PATH = "../data/wikidata_qid_titles.csv"
INDEX_PATH = "../candidates/index/bm25.pkl"

DENSE_INDEX_DIR = "../candidates/index/"  
DENSE_MODEL     = "Qwen/Qwen3-Embedding-4B"

BENCHMARKS = "../data/benchmarks"
RESULTS_DIR = "../evaluation/results"
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

# ELGOLD Dataset

Files `.txt` with annotations `{{mention|TYPE|gold_title}}`.
We extract each mention + context (150 chars from each side).

In [9]:
df = load_benchmark(BENCHMARKS, MAX_EXAMPLES)

print(f"Exemples : {len(df)}")
print(f"Fichiers : {df['file'].nunique()}")
print(f"Types : {df['entity_type'].value_counts().to_dict()}")
df.head(3)

Exemples : 3103
Fichiers : 260
Types : {'GPE': 625, 'PERSON': 588, 'ORG': 450, 'SUBSTANCE': 264, 'SPECIE': 211, 'DISEASE': 205, 'PRODUCT': 200, 'WORK_OF_ART': 180, 'NORP': 121, 'LOC': 95, 'EVENT': 75, 'FAC': 63, 'LAW': 14, 'LANGUAGE': 12}


,file,mention,entity_type,gold,context
0,5_14.txt,Hollywood,ORG,Cinema of the United States,is a Bad movie from start to finish. Perhaps t...
1,5_14.txt,Hellbound,WORK_OF_ART,The Hellbound Heart,ime for Hollywood to find some fresh material ...
2,5_14.txt,Constantine,WORK_OF_ART,Constantine (film),good. (If your into catholic guilt that is!) ...


# Candidate generation

## Sparse retrieval (BM25)

### BM25 index

The index is built the first time (only once) from `wikidata_qid_titles.csv` and saved as `.pkl`.  
The following times, we load it from the `.pkl`  

The index contains Wikipedia titles (candidate entities).  

In [10]:
if Path(INDEX_PATH).exists():
    print("Index found → loading...")
    retriever = BM25Retriever.load(INDEX_PATH)
else:
    print("Index not found → construction...")
    titles_df = pd.read_csv(TITLES_PATH)
    print(f"Columns : {titles_df.columns.tolist()}")

    titles = titles_df["title"].dropna().tolist()
    print(f"{len(titles):,} titles loaded")
    retriever = BM25Retriever.from_entity_list(titles)
    retriever.save(INDEX_PATH)

print(f"Index ready : {len(retriever.entity_titles):,} entities")

Index found → loading...
[BM25] Index loaded from ../candidates/index/bm25.pkl (7517498 entities)
Index ready : 7,517,498 entities


BM25 verification (small test)

In [11]:
for _, row in df.head(5).iterrows():
    candidates = retriever.retrieve(row["mention"], k=K)
    gold_found = "✓" if recall_at_k(row["gold"], candidates) else "✗"
    print(f"{gold_found} mention='{row['mention']}'  gold='{row['gold']}'")
    print(f"  → {candidates}\n")

✗ mention='Hollywood'  gold='Cinema of the United States'
  → ['Hollywood Hollywood', 'Hollywood', 'West Hollywood (Hollywood, Florida)', 'Hollywood Vampires (Hollywood Vampires album)', 'Hollywood cycles', 'Hollywood Squares', 'Hollywood, Alabama', 'Hollywood, Florida', 'DJ Hollywood', 'Hollywood Bowl', 'Hollywood North', 'Hollywood Science', 'Hollywood Freeway', 'Hollywood Ending', 'Hollywood Records', 'Hollywood Pictures', 'Hollywood Video', 'Hollywood, Worcestershire', 'Hollywood Boulevard', 'Hollywood Heartbreakers', 'Hollywood Hotel', 'Hollywood Hills', 'Hollywood Rose', 'Access Hollywood', 'New Hollywood', 'Hollywood, Maryland', 'Planet Hollywood', 'Hollywood Palladium', 'North Hollywood', 'Jimmy Hollywood', 'Hollywood station', 'Hollywood Babylon', 'Hollywood Sign', 'Hollywood Stars', 'Plaza Hollywood', 'Hollywood novel', 'Hollywood Hijinx', 'Hollywood Airport', 'Hollywood Star', 'Ovation Hollywood', 'Hollywood Blonds', 'Hollywood Road', 'Hollywood Showdown', 'Doc Hollywood', '

### Recall@K on benchmark

Recall@K = pourcentage d'exemples où le gold est dans les K candidats BM25.  

L'accuracy ne peut pas le dépasser.

In [ ]:
# Generate candidates for all the benchmark
df["candidates"] = df["mention"].apply(lambda m: retriever.retrieve(m, k=K))
df["gold_in_candidates"] = df.apply(
    lambda row: recall_at_k(row["gold"], row["candidates"]), axis=1
)

recall = df["gold_in_candidates"].mean()
print(f"Recall@{K} (BM25) : {recall:.2%}")

missed = df[~df["gold_in_candidates"]]
print(f"\nGold absent des candidats : {len(missed)}/{len(df)} exemples")
print("Exemples ratés par BM25 :")
missed[["mention", "gold", "candidates"]].head(5)

## Dense retrieval

In [ ]:
dense_index_exists = (
    os.path.exists(f"{DENSE_INDEX_DIR}/dense.faiss") and
    os.path.exists(f"{DENSE_INDEX_DIR}/dense_meta.pkl")
)
 
if dense_index_exists:
    print("Dense index found → loading ...")
    dense_retriever = DenseRetriever.load(DENSE_INDEX_DIR)
else:
    print("Dense index not found → building ...")

    # liste déjà chargée dans retriever.entity_titles (BM25)
    titles = retriever.entity_titles  
    dense_retriever = DenseRetriever(model_name=DENSE_MODEL)
    dense_retriever.build_index(titles, batch_size=64)
    dense_retriever.save(DENSE_INDEX_DIR)
 
print(f"Dense index ready : {dense_retriever.index.ntotal:,} vectors")

Dense retrieval check

In [ ]:
for _, row in df.head(5).iterrows():
    candidates_d = dense_retriever.retrieve(row["mention"], k=K)
    found = "✓" if recall_at_k(row["gold"], candidates_d) else "✗"
    print(f"{found}  '{row['mention']}'  →  gold='{row['gold']}'")
    print(f"     {candidates_d}")
    print()

## Recall@K

In [ ]:
def fuse_candidates(bm25_cands: list[str], dense_cands: list[str], k: int = 10) -> list[str]:
    seen, merged = set(), []
    for b, d in zip(bm25_cands, dense_cands):
        for c in (b, d):
            if c.lower() not in seen:
                seen.add(c.lower())
                merged.append(c)
            if len(merged) == k:
                return merged
    # If one list is shorter, drain the other
    for c in (bm25_cands + dense_cands):
        if c.lower() not in seen:
            seen.add(c.lower())
            merged.append(c)
        if len(merged) == k:
            return merged
    return merged[:k]

In [ ]:
# Generate candidates for all 3 modes
df["candidates_bm25"] = df["mention"].apply(lambda m: retriever.retrieve(m, k=K))
df["candidates_dense"] = df["mention"].apply(lambda m: dense_retriever.retrieve(m, k=K))
df["candidates_fusion"] = df.apply(
    lambda row: fuse_candidates(row["candidates_bm25"], row["candidates_dense"], k=K),
    axis=1
)

In [ ]:
# Recall@K for each
recall_bm25 = df.apply(lambda r: recall_at_k(r["gold"], r["candidates_bm25"]), axis=1).mean()
recall_dense = df.apply(lambda r: recall_at_k(r["gold"], r["candidates_dense"]), axis=1).mean()
recall_fusion = df.apply(lambda r: recall_at_k(r["gold"], r["candidates_fusion"]), axis=1).mean()
 
print(f"Recall@{K}  BM25    : {recall_bm25:.2%}")
print(f"Recall@{K}  Dense   : {recall_dense:.2%}")
print(f"Recall@{K}  Fusion  : {recall_fusion:.2%}  ← expected best")
print()
print("Plafonds zero-shot (le LLM ne peut pas dépasser ces chiffres) :")
print(f"  BM25   → max accuracy = {recall_bm25:.2%}")
print(f"  Dense  → max accuracy = {recall_dense:.2%}")
print(f"  Fusion → max accuracy = {recall_fusion:.2%}")

# Evaluation

## Loading model

In [13]:
tokenizer, model = load_model(MODEL_NAME)
print(f"Loaded on : {next(model.parameters()).device}")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|███████████████████████████████████████| 3/3 [00:45<00:00, 15.14s/it]


Loaded on : cuda:0


In [15]:
def run_eval_mode(df_in, candidates_col, mode_label):
    records = df_in.to_dict("records")
    for rec in records:
        rec["candidates"] = rec[candidates_col]
        rec["gold_in_candidates"] = recall_at_k(rec["gold"], rec["candidates"])

    all_cands = set(c for r in records for c in r["candidates"])
    fetch_descriptions_batch(list(all_cands))

    results = []
    for i in tqdm(range(0, len(records), BATCH_SIZE), desc=mode_label):
        batch = records[i : i + BATCH_SIZE]
        prompts = [
            build_mcq_prompt(r["context"], r["mention"], r["entity_type"],
                              r["candidates"], descriptions=fetch_descriptions_batch(r["candidates"]))
            for r in batch
        ]
        try:
            raws = generate_batch(prompts, tokenizer, model, MAX_NEW_TOKENS)
        except Exception as e:
            for r in batch:
                results.append({**r, "predicted": "", "correct": False, "raw_output": "", "error": str(e)})
            continue

        for rec, raw in zip(batch, raws):
            try:
                parsed = parse_mcq_response(raw, candidates=rec["candidates"])
                pred, error = parsed["predicted"], ""
            except Exception as e:
                pred, error = "", str(e)
            results.append({
                "mention": rec["mention"], "gold": rec["gold"],
                "candidates": rec["candidates"],
                "gold_in_candidates": rec["gold_in_candidates"],
                "predicted": pred,
                "correct": is_correct(pred, rec["gold"]) if not error else False,
                "raw_output": raw, "error": error, "retrieval_mode": mode_label,
            })
    return pd.DataFrame(results)

In [16]:
slug = MODEL_NAME.replace("/", "_")

df_bm25 = run_eval_mode(df, "candidates_bm25", "zero_shot_bm25", tokenizer, model)
df_dense  = run_eval_mode(df, "candidates_dense", "zero_shot_dense", tokenizer, model)
df_fusion = run_eval_mode(df, "candidates_fusion", "zero_shot_fusion", tokenizer, model)
 
# Save
for df_r, label in [(df_bm25, "zero_shot_bm25"), (df_dense, "zero_shot_dense"), (df_fusion, "zero_shot_fusion")]:
    df_r.to_csv(f"{RESULTS_DIR}/{label}_{slug}.csv", index=False)

KeyError: 'candidates_bm25'

In [ ]:
import pandas as pd
 
rows = []
for df_r, label, recall in [
    (df_bm25, "BM25", recall_bm25),
    (df_dense, "Dense", recall_dense),
    (df_fusion, "Fusion", recall_fusion),
]:
    m = compute_metrics(df_r)
    rows.append({
        "Retriever": label,
        "Recall@K": f"{recall:.2%}",
        "Accuracy": f"{m['accuracy']:.2%}",
        "LLM loss": f"-{(recall - m['accuracy']):.2%}",
        "n_correct": m["n_correct"],
        "n_total": m["n_total"],
    })
 
df_comparison = pd.DataFrame(rows)
print(f"\nModel : {MODEL_NAME}")
print(df_comparison.to_string(index=False))

## Test on examples

In [ ]:
for rec in df.head(3).to_dict("records"):
    descs = fetch_descriptions_batch(rec["candidates"])
    prompt = build_mcq_prompt(rec["context"], rec["mention"], rec["entity_type"],
                               rec["candidates"], descriptions=descs)
    raw = generate_batch([prompt], tokenizer, model)[0]
    parsed = parse_mcq_response(raw, candidates=rec["candidates"])
    ok = is_correct(parsed["predicted"], rec["gold"])
    print(f"{'✓' if ok else '✗'}  mention='{rec['mention']}'")
    print(f"   gold : {rec['gold']}")
    print(f"   raw  : {repr(raw[:200])}")
    print(f"   pred : {parsed['predicted']}\n")

## Zero-shot Evaluation

In [ ]:
results = []
records = df.to_dict("records")

# Pre-fetch descriptions
all_cands = set(c for rec in records for c in rec["candidates"])
print(f"Pre-fetching {len(all_cands):,} descriptions...")
fetch_descriptions_batch(list(all_cands))

for i in tqdm(range(0, len(records), BATCH_SIZE)):
    batch = records[i : i + BATCH_SIZE]

    prompts = [
        build_mcq_prompt(r["context"], r["mention"], r["entity_type"],
                          r["candidates"], descriptions=fetch_descriptions_batch(r["candidates"]))
        for r in batch
    ]

    try:
        raws = generate_batch(prompts, tokenizer, model, MAX_NEW_TOKENS)
    except Exception as e:
        for r in batch:
            results.append({**r, "predicted": "", "correct": False, "raw_output": "", "error": str(e)})
        continue

    for rec, raw in zip(batch, raws):
        try:
            parsed = parse_mcq_response(raw, candidates=rec["candidates"])
            pred, error = parsed["predicted"], ""
        except Exception as e:
            pred, error = "", str(e)

        results.append({
            "file": rec["file"], "mention": rec["mention"],
            "entity_type": rec["entity_type"], "gold": rec["gold"],
            "candidates": rec["candidates"],
            "gold_in_candidates": rec["gold_in_candidates"],
            "predicted": pred,
            "correct": is_correct(pred, rec["gold"]) if not error else False,
            "raw_output": raw,
            "error": error,
        })

df_results = pd.DataFrame(results)
df_results.to_csv(f"{RESULTS_DIR}/{MODE}_{MODEL_NAME.replace('/', '_')}.csv", index=False)

## Results

In [ ]:
metrics = compute_metrics(df_results)
print_metrics(metrics, model_name=f"{MODE} / {MODEL_NAME}")

if MODE == "zero_shot_bm25":
    print(f"\nRecall@{K} BM25 (plafond) : {recall:.2%}")
    print(f"Gap plafond → accuracy : -{(recall - metrics['accuracy']):.2%}  (LLM errors)")

# Errors: when gold was among candidates but the LLM did not select it
ok_retrieval = df_results[df_results["gold_in_candidates"]] if "gold_in_candidates" in df_results.columns else df_results
llm_errors = ok_retrieval[~ok_retrieval["correct"] & (ok_retrieval["error"] == "")]
print(f"\nErreurs LLM (gold était dans les candidats) : {len(llm_errors)}")
for _, r in llm_errors.head(5).iterrows():
    print(f"  '{r['mention']}' | gold='{r['gold']}' | pred='{r['predicted']}' | raw={repr(r['raw_output'])}")